In [178]:
import random

import pandas as pd
from pandas import DataFrame
import numpy as np

# startPoint = {"lat": 30.317504,"lon": 59.927085}
# endPoint = {"lat": 30.327108, "lon": 59.935408}

POPULATION_SIZE = 3
GENERATIONS = 100

MIN_ROUTE_POINTS = 2
MAX_ROUTE_POINTS = 10

random.seed(42)
np.random.seed(42)

# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }


In [179]:
df = pd.read_csv('data/places.csv')

**Алгоритмы генетики**

In [180]:
from haversine import haversine

# Оптимизация конкретного маршрута
def optimize_route(route_ids, df, startPoint):
    print("Маршрут до оптимизации:", route_ids)
    points = (
        df[df["id"].isin(route_ids)]
        [["id", "lat", "lon"]]
        .copy()
    )

    remaining = points.to_dict("records")

    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    optimized_route = []

    while remaining:
        nearest = min(
            remaining,
            key=lambda p: haversine(
                (current["lat"], current["lon"]),
                (p["lat"], p["lon"])
            )
        )

        optimized_route.append(nearest["id"])

        current = nearest

        remaining.remove(nearest)

    print("Маршрут после оптимизации:", optimized_route)
    return optimized_route

# Генерация случайной особи
def create_population(df: DataFrame, startPoint: dict[str, float], endPoint: dict[str, float], num: int):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    buffer_km = 0.5

    # Переводим километры в градусы
    lat_buffer = buffer_km / 111

    mean_lat = (startPoint["lat"] + endPoint["lat"]) / 2
    lon_buffer = buffer_km / (111 * np.cos(np.radians(mean_lat)))

    min_lat = min(startPoint["lat"], endPoint["lat"]) - lat_buffer
    max_lat = max(startPoint["lat"], endPoint["lat"]) + lat_buffer

    min_lon = min(startPoint["lon"], endPoint["lon"]) - lon_buffer
    max_lon = max(startPoint["lon"], endPoint["lon"]) + lon_buffer

    df_filtered = df[
        (df["lat"] >= min_lat) &
        (df["lat"] <= max_lat) &
        (df["lon"] >= min_lon) &
        (df["lon"] <= max_lon)
    ]

    population = []

    df_indexes = df_filtered["id"].tolist()

    for _ in range(num):
        route_id = random.sample(
            range(len(df_indexes)),
            route_size
        )
        route = []

        for i in route_id:
            route.append(df_indexes[i])
        
        print("======", _, "======")
        points = (
            df[df["id"].isin(route)]
            [["id", "name"]]
            .copy()
        )
        for i in points.values:
            print(i)
        
        new_route = optimize_route(route, df, startPoint)
        
        print("======", _, "======")
        points = (
            df[df["id"].isin(route)]
            [["id", "name"]]
            .copy()
        )
        for i in points.values:
            print(i)
        population.append({"route": new_route, "fitness": None})

    return population

**Запуск генетики**

In [181]:
startPoint = {"lat": 59.927085,"lon": 30.317504}
endPoint = {"lat": 59.935408, "lon": 30.327108}

population = create_population(df, startPoint, endPoint, POPULATION_SIZE)


points = (
    df[df["id"].isin(population[1]["route"])]
    [["id", "name"]]
    .copy()
)

for i in points.values:
    print(i)

====== 0 ======
[1464061393 'Метеорологический павильон']
[4915782458 'Соната']
[11914860369 'Место-сюрприз']
Маршрут до оптимизации: [1464061393, 11914860369, 4915782458]
Маршрут после оптимизации: [11914860369, 1464061393, 4915782458]
====== 0 ======
[1464061393 'Метеорологический павильон']
[4915782458 'Соната']
[11914860369 'Место-сюрприз']
====== 1 ======
[3842754772 'Флора Фарнезская']
[4690381015 'Грибоедов хаус']
[4817904721 'Apartment on Moyka 40']
Маршрут до оптимизации: [4817904721, 4690381015, 3842754772]
Маршрут после оптимизации: [4690381015, 4817904721, 3842754772]
====== 1 ======
[3842754772 'Флора Фарнезская']
[4690381015 'Грибоедов хаус']
[4817904721 'Apartment on Moyka 40']
====== 2 ======
[2707562249 'Абажур']
[10279481710 'На Речке']
[11696663969 'Catherine Art Hotel']
Маршрут до оптимизации: [11696663969, 2707562249, 10279481710]
Маршрут после оптимизации: [10279481710, 2707562249, 11696663969]
====== 2 ======
[2707562249 'Абажур']
[10279481710 'На Речке']
[116966

In [182]:
import folium

def hex_to_rgb(hex_str):
    """Преобразует HEX в кортеж RGB (0-255)"""
    hex_str = hex_str.lstrip('#')
    return tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(rgb):
    """Преобразует RGB в HEX строку"""
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))

def generate_gradient(start_hex, end_hex, steps):
    """Генерирует список цветов для градиента"""
    start_rgb = hex_to_rgb(start_hex)
    end_rgb = hex_to_rgb(end_hex)
    
    gradient_colors = []
    for i in range(steps):
        t = i / max(1, steps - 1)
        r = start_rgb[0] + (end_rgb[0] - start_rgb[0]) * t
        g = start_rgb[1] + (end_rgb[1] - start_rgb[1]) * t
        b = start_rgb[2] + (end_rgb[2] - start_rgb[2]) * t
        gradient_colors.append(rgb_to_hex((r, g, b)))
        
    return gradient_colors

path_colors = generate_gradient("#FF0000","#0000FF", len(population))

route_df = df[(df["id"].isin(population[0]["route"]))]
m = folium.Map(
    location=[
        route_df["lat"].mean(),
        route_df["lon"].mean()
    ],
    zoom_start=15
)

for i in range(len(population)):
    individual = population[i]
    route_df = df[(df["id"].isin(individual["route"]))]

    # Точки маршрута
    folium.Marker(
        [startPoint["lat"], startPoint["lon"]]
    ).add_to(m)
    folium.Marker(
        [endPoint["lat"], endPoint["lon"]]
    ).add_to(m)

    route_line = [[startPoint["lat"], startPoint["lon"]]]
    route_line.extend(route_df[["lat", "lon"]].values.tolist())
    route_line.append([endPoint["lat"], endPoint["lon"]])

    # Линия маршрута
    folium.PolyLine(
        route_line,
        weight=4,
        color=path_colors[i]
    ).add_to(m)

m.save(r"visualisations\route.html")